# 0. Setting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/main/

In [ ]:
! pip install torch-geometric --quiet

# 1. Data Generation

In [ ]:
# 1-1. Baseline Dataset Generation
! python -m src.data.multiplex_generator_v3 \
  --size 1500 \
  --seed 1024 \
  --out_dir data/multiplex_base \
  --config configs/generator_baseline.json

In [ ]:
# 1-2. Ontology Validation
! python -m src.cli.validate_ontology \
  --manifest data/multiplex_base/multiplex.json \
  --ontology ontology/terror.ttl \
  --shapes ontology/constraints.shacl.ttl \
  --json

In [ ]:
# 1-3. PyG Dataset Transformation
! python -m src.data.build_pyg_dataset_v3 \
  --manifest data/multiplex_base/multiplex.json \
  --out_path data/multiplex_base/pyg_data.pt

In [ ]:
# 1-4. Data Statistics
! python -m src.data.basic_diagnostics_v3 \
  --manifest data/multiplex_base/multiplex.json \
  --out_dir data/analysis/multiplex_baseline

# 2. End-To-End (Run all)
 - Run All : Data Generation + Ontology validation + PyG + Statistics


In [ ]:
# 2-1. Strict mode
# base
! python -m src.run_all \
  --config configs/generator_baseline.json \
  --size 1500 \
  --seed 2025 \
  --out_root data/multiplex_base

# hard
! python -m src.run_all \
  --config configs/generator_hard.json \
  --size 1500 \
  --seed 2025 \
  --out_root data/multiplex_hard

# easy
! ! python -m src.run_all \
  --config configs/generator_easy.json \
  --size 1500 \
  --seed 2025 \
  --out_root data/multiplex_easy

In [ ]:
# 2-2. Constrained mode

# base
! python -m src.run_all \
  --config configs/generator_baseline.json \
  --size 1500 \
  --seed 2025 \
  --out_root data/multiplex_base \
  --ontology_mode constrained

# hard
! python -m src.run_all \
  --config configs/generator_hard.json \
  --size 1500 \
  --seed 2025 \
  --out_root data/multiplex_hard \
  --ontology_mode constrained

# easy
! python -m src.run_all \
  --config configs/generator_easy.json \
  --size 1500 \
  --seed 2025 \
  --out_root data/multiplex_easy \
  --ontology_mode constrained

In [ ]:
# 2-3. Report Only mode
! python -m src.run_all \
  --config configs/generator_baseline.json \
  --size 800 \
  --seed 2025 \
  --out_root results \
  --ontology_mode report_only

# 3. GNN Models Solution

## 3-1. High-Value Target(HVT) Classification

In [ ]:
# 3-1-1. HVT Classification

! python -m src.models.train_hvt_gnn_v3 \
  --data_path data/multiplex_base/run_20260215T063746Z_a637982d_seed2025/pyg_data.pt \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --pos_weight 10 \
  --edge_attr_agg --edge_attr_transform none \
  --include_edge_flags


! python -m src.models.train_hvt_gnn_v3 \
  --data_path data/multiplex_easy/run_20260215T064034Z_db18c07e_seed2025/pyg_data.pt \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --pos_weight 10 \
  --edge_attr_agg --edge_attr_transform none \
  --include_edge_flags


! python -m src.models.train_hvt_gnn_v3 \
  --data_path data/multiplex_hard/run_20260215T063909Z_1c41c1f6_seed2025/pyg_data.pt \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --pos_weight 10 \
  --edge_attr_agg --edge_attr_transform none \
  --include_edge_flags

## 3-2. Multi-Tasking
 - Role Classification
 - HVT Classification
 - importance score Regression

In [ ]:
# 3-2. Multi-Tasking
! python -m src.models.train_multitask_gnn_v3 \
  --data_path data/multiplex_base/run_20260215T063746Z_a637982d_seed2025/pyg_data.pt \
  --encoder transformer \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 2e-3 --weight_decay 1e-4 --epochs 300 \
  --seed 2025 --patience 50 \
  --edge_attr_transform none \
  --include_edge_flags


! python -m src.models.train_multitask_gnn_v3 \
  --data_path data/multiplex_easy/run_20260215T064034Z_db18c07e_seed2025/pyg_data.pt \
  --encoder transformer \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 2e-3 --weight_decay 1e-4 --epochs 300 \
  --seed 2025 --patience 50 \
  --edge_attr_transform none \
  --include_edge_flags


! python -m src.models.train_multitask_gnn_v3 \
  --data_path data/multiplex_hard/run_20260215T063909Z_1c41c1f6_seed2025/pyg_data.pt \
  --encoder transformer \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 2e-3 --weight_decay 1e-4 --epochs 300 \
  --seed 2025 --patience 50 \
  --edge_attr_transform none \
  --include_edge_flags

## 3-3. Specific node link prediction
 - Finance layer illegal fund link prediction
 - Communication Layer Contact Prediction


In [ ]:
# 3-3-1. Finance layer prediction
! python -m src.models.train_linkpred_layer_v3 \
  --data_path data/multiplex_base/run_20260215T063746Z_a637982d_seed2025/pyg_data.pt \
  --layer finance \
  --hidden_dim 192 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags


! python -m src.models.train_linkpred_layer_v3 \
  --data_path data/multiplex_easy/run_20260215T064034Z_db18c07e_seed2025/pyg_data.pt \
  --layer finance \
  --hidden_dim 192 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags


! python -m src.models.train_linkpred_layer_v3 \
  --data_path data/multiplex_hard/run_20260215T063909Z_1c41c1f6_seed2025/pyg_data.pt \
  --layer finance \
  --hidden_dim 192 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags

In [ ]:
# 3-3-2. Communication Layer prediction
! python -m src.models.train_linkpred_layer_v3 \
  --data_path data/multiplex_base/run_20260215T063746Z_a637982d_seed2025/pyg_data.pt \
  --layer communication \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags


! python -m src.models.train_linkpred_layer_v3 \
  --data_path data/multiplex_easy/run_20260215T064034Z_db18c07e_seed2025/pyg_data.pt \
  --layer communication \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags


! python -m src.models.train_linkpred_layer_v3 \
  --data_path data/multiplex_hard/run_20260215T063909Z_1c41c1f6_seed2025/pyg_data.pt \
  --layer communication \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags

# 4. Result Visualization

In [ ]:
# Evaluation Summary and Plots
! python -m src.analysis.plot_multitask_linkpred_summary \
  --run_dirs data/multiplex_base/run_20260215T063746Z_a637982d_seed2025/pyg_data.pt \
  --out_dir results/summary_all \
  --difficulty_mode auto \
  --aggregate \
  --save_runs_csv \
  --write_benchmark_table


! python -m src.analysis.plot_multitask_linkpred_summary \
  --run_dirs data/multiplex_easy/run_20260215T064034Z_db18c07e_seed2025 \
  --out_dir results/summary_all \
  --difficulty_mode auto \
  --aggregate \
  --save_runs_csv \
  --write_benchmark_table


! python -m src.analysis.plot_multitask_linkpred_summary \
  --run_dirs data_path data/multiplex_hard/run_20260215T063909Z_1c41c1f6_seed2025 \
  --out_dir results/summary_all \
  --difficulty_mode auto \
  --aggregate \
  --save_runs_csv \
  --write_benchmark_table